# 1.2 PyTorch for Computer Vision

In this note, we will address some of the basic commands that you will need from PyTorch in general and in computer vision in particular.

First, we import the required libraries. These should all be available on Google Colab.

In [ ]:
import torch # PyTorch
import torchvision # Computer vision
# Data sets and transforms
from torchvision import datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
# Other libraries
import matplotlib.pyplot as plt
import numpy as np

print("PyTorch version:", torch.__version__)

## PyTorch Tensors

A tensor is an array of numbers. All data will have to be put in this format.

In [ ]:
scalar = torch.tensor(3)
vector = torch.tensor([1., 2.])

A basic tensor has three attributes you will likely interact with:

- **shape** — how the data are arranged
- **dtype** — how each value is represented, such as `float32` or `int64`
- **device** — where the data live, usually the CPU or a GPU - this will become relevant when training on PyTorch model on a GPU

<!-- Floating-point tensors are normally used for model inputs and parameters. Integer tensors are often used for class labels or indices. -->

In [ ]:
print('shape:  ', vector.shape)
print('dtype:  ', vector.dtype)
print('device: ', vector.device)

Many of the commands (such as the method creating vectors with ones in each entry) mirror the corresponding commands in numpy.

In [ ]:
t0 = torch.zeros(size=(3,1))
print(t0)
t1 = torch.ones(size=(3,1))
print(t1)

Just like in numpy you can, for instance, easily perform element-wise multiplication, multiply a tensor with a scalar or determine the index of the entry with the maximum or minimum value. But not everything is the same.

In [ ]:
print(vector)
print(10 * vector)
print(vector * vector)
print(vector.argmax())
print(vector.argmin())

When creating so-called batches (see Lecture 1) of data in later notebooks, we will sometimes need to be able to add a dimension to a tensor. We thus keep the content of the data but add an empty dimension along the dimension dim.



In [ ]:
x = torch.tensor([2, -1, 3])
print(x, ' has Shape: ', x.shape)

# add a dimension to the tensor
x = x.unsqueeze(dim=0)
print(x, ' has Shape: ', x.shape)



PyTorch convention puts the **batch dimension first**, i.e. given an input 
$
X \in \mathbb{R}^{n\times m},
$
then $n$ is the batch size and $m$ is input dimension.

We can also turn PyTorch tensors into numpy arrays:

In [ ]:
vector.numpy()

or numpy arrays into tensors

In [ ]:
x = np.arange(4)
print(x)
print(torch.tensor(x))

The exercises at the end of this notebook cover more tensor operations that will be useful for transforming data.

## Computer Vision Datasets 

In this course, we will dive into Deep Learning for computer vision, i.e., we will be working with image data. Torchvision provides a long list of datasets that are commonly used for this purpose.

In [ ]:
# Available datasets
print(datasets.__all__)

In Notebook 1.1 we downloaded the MNIST dataset using torchvision:

In [ ]:
trainset_mnist_0 = datasets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())

Note that not all datasets listed above can be easily downloaded with one line of code.

We can plot individual images from this data set using matplotlib. Each image is a PyTorch tensor object.

In [ ]:
# This visualises random samples from the training dataset
fig, ax = plt.subplots(1, 5, figsize=(15, 3))

# select 5 random images to display
random_idx = torch.randint(low=0,high=len(trainset_mnist_0),size=(5,))

for i, idx in enumerate(random_idx):
    img = trainset_mnist_0.data[idx]
    ax[i].imshow(img, cmap='gray')
    ax[i].axis('off')

Let's try a different dataset...

In [ ]:
# The CIFAR dataset is hosted by the University of Toronto and the download 
# speed is very slow, so I have included an alternative mirror.
!mkdir -p data
!wget -c https://data.brainchip.com/dataset-mirror/cifar10/cifar-10-python.tar.gz \
    -O data/cifar-10-python.tar.gz
!tar -xzf data/cifar-10-python.tar.gz -C data

In [ ]:
trainset_cifar10_0 = datasets.CIFAR10(root='./data', train=True, download=False, transform=transforms.ToTensor())

In [ ]:
fig, ax = plt.subplots(1, 5, figsize=(15, 3))
random_idx = torch.randint(low=0,high=len(trainset_cifar10_0),size=(5,))
for i, idx in enumerate(random_idx):
    img = trainset_cifar10_0.data[idx]
    ax[i].imshow(img, cmap='gray')
    ax[i].axis('off')

### Colour Channels
The CIFAR dataset contains colour images. When we load the images into the notebook (from whatever compressed format they are in), the colours are representated by three channels, red (R), green (G) and blue (B). Let's print the shape of the MNIST and CIFAR images:


In [ ]:
(mnist_img, _), (cifar_img, _) = trainset_mnist_0[0], trainset_cifar10_0[0]
print('MNIST (grayscale): ', mnist_img.shape)
print('CIFAR (RGB):       ', cifar_img.shape)

## Transforms

Before we can pass the data to our Deep Learning model, we will often have to transform it to ensure that it has the correct format. Maybe the images are too small or too large compared to the images on which we have trained our model. It is also common practice to normalise the images before passing them to the Deep Learning model, as this approach is beneficial for convergence, stability during training and the generalisation of the network.

Let's define a transform for normalisation.

In [ ]:
# MNIST is black and white. So, we only need one color channel. Other data sets have three 
means = (0.5)
stds = (0.5)

# Define the data transform including normalization
transform_mnist_0 = transforms.Compose([
    transforms.ToTensor(),  # convert to PyTorch tensor
    transforms.Normalize(mean=means, std=stds)  # normalize images
])

# In contrast, CIFAR10 provides pictures with three channels (RGB). 
means10 = (0.5,0.5,0.5)
stds10 = (0.5,0.5,0.5)

# Define the data transform including normalization
transform_cifar10_0 = transforms.Compose([
    transforms.ToTensor(),  # Converts the image to a PyTorch tensor
    transforms.Normalize(means10, stds10),  # Normalize to a range of -1 to 1 for three channels
])

If we had defined the transform before we loaded the data, we could have applied the transform to the data right away. However, we can still do so retrospectively.

In [ ]:
# Apply the transform to the dataset
trainset_cifar10_0.transform = transform_cifar10_0

We can divide the entire dataset into smaller batches, shuffle it and provide an iterable interface for retrieving batches during the training process using the DataLoader class. (see Notebook 1.3 for more details)

In [ ]:
train_loader_cifar10 = DataLoader(trainset_cifar10_0, batch_size=32, shuffle=True)

If you plot the image, you can now see that the colours are a bit off because we normalised the image.

In [ ]:
# Get the first batch from the data loader
dataiter = iter(train_loader_cifar10)
images, labels = next(dataiter)

fig, ax = plt.subplots(1, 5, figsize=(15, 3))
for i in range(5):
    img = images[i].permute(1, 2, 0)
    ax[i].imshow(img)
    ax[i].axis('off')

We can also use transforms for image augmentation, i.e., we scale, translate or rotate the images to create new images. By doing so, we reduce overfitting and introduce scale and orientation invariance.

In [ ]:
transform_mnist_1 = transforms.Compose([
    transforms.RandomHorizontalFlip(), # Randomflip
    transforms.RandomRotation(degrees=(-90, 90)),  # Random rotation between -90 and 90 degrees
    transforms.ToTensor(),  # convert to PyTorch tensor
    transforms.Normalize(mean=means, std=stds)  # normalize images
])

# Apply the transform to the dataset
trainset_mnist_0.transform = transform_mnist_1
train_loader_mnist = DataLoader(trainset_mnist_0, batch_size=32, shuffle=True)

In [ ]:
# Get the first batch from the data loader
dataiter = iter(train_loader_mnist)
images, labels = next(dataiter)

fig, ax = plt.subplots(1, 5, figsize=(15, 3))
for i in range(5):
    img = images[i].permute(1, 2, 0)
    ax[i].imshow(img, cmap="gray")
    ax[i].axis('off')

**Exercise 1**: Find out how to `transpose`, `stack`, `reshape` and `permute` tensors.

**Exercise 2**: Construct a matrix as a PyTorch tensor. Perform matrix multiplication (cf. torch.matmul())

**Exercise 3**: Just like you can `unsqueeze` a tensor, you can `squeeze` it. Find out how and what it means.

**Exercise 4**: Load and plot examples from another data set from torchvision.

**Exercise 5**: In cell [18] above, where we plot the CIFAR10 images after normalisation, the colours are off. Correct for this by denormalising the picture before plotting.

**Exercise 6**: Above, we discuss a few transforms for data augmentation. What other transforms are there for this purpose? Write the corresponding code below.